# Lab W7: LLM-Assisted Feature Development

PI meminta: "tambahkan mixup augmentation ke training loop classifier sentimen, lalu bandingkan dengan baseline". Kamu mengimplementasi `mixup_batch` lewat protokol 5-tahap LLM - pseudocode sendiri dulu, lalu prompt, output verbatim, verifikasi, sanity test - bukan menyalin mentah dari LLM. Ada satu twist domain teks: token adalah ID diskret yang tidak bisa diinterpolasi, jadi mixup harus diterapkan di **level embedding** (vektor kontinu), bukan di token. Konsep verifikasi dan synthesis rule mengacu ke `07_W7_Text_Transformers_Repo_Adoption.md` §2; sifat token vs embedding mengacu §1.2-§1.3.

**Prasyarat:** Bab W7 §1.2-§1.3 (tokenization, embedding) dan §2 (verifikasi rule, synthesis 2 sumber, AI untuk non-kode) sudah dibaca, familiar dengan konsep Mixup (Zhang et al. 2018, arXiv:1710.09412), dan punya akses ke satu LLM (Claude/ChatGPT/Copilot). **Hardware & waktu:** CPU cukup (classifier teks ringan, 2 seed x 15 epoch ~2-5 menit total) atau GPU, total ~3-4 jam termasuk protokol LLM. Butuh koneksi internet untuk mengunduh dataset SmSA.

## Alur Lab

1. **Pseudocode manual:** tulis perilaku `mixup_batch` sebelum bertanya ke LLM.
2. **Prompt mentah:** simpan prompt persis sebagai bukti proses.
3. **Output mentah:** simpan jawaban LLM sebelum modifikasi.
4. **Fungsi kecil:** implementasikan `mixup_batch` dan `mixup_criterion`.
5. **Sanity tests:** cek pass-through, shape, nilai lambda, dan loss.
6. **Eksperimen kecil:** bandingkan baseline vs mixup (level embedding) pada SmSA dengan 2 seed.
7. **Log proses:** tulis apa yang diubah dari output LLM.

## 0. Setup

Sel ini memuat dataset sentimen SmSA langsung dari TSV mentah, membangun tokenizer word-level sederhana (vocab dari data train), lalu mendefinisikan classifier teks ringan: embedding -> mean-pool -> head. Pemisahan `encode` (menghasilkan vektor kalimat kontinu) dan `classify` (head) penting, karena mixup nanti disisipkan di antara keduanya.

In [ ]:
import time
from pathlib import Path
from datetime import date
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset

ROOT = Path("..").resolve()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 1. Muat SmSA (sentimen Bahasa Indonesia) langsung dari TSV mentah ---
base_url = "https://raw.githubusercontent.com/IndoNLP/indonlu/master/dataset/smsa_doc-sentiment-prosa/"
labels = ["negatif", "positif", "netral"]
label_map = {"positive": 1, "positif": 1, "neutral": 2, "netral": 2, "negative": 0, "negatif": 0}
numeric_map = {"0": 1, "1": 2, "2": 0}  # cadangan kalau label tersimpan sebagai angka

def load_smsa(split):
    fname = "valid_preprocess.tsv" if split == "valid" else f"{split}_preprocess.tsv"
    df = pd.read_csv(base_url + fname, sep="\t", names=["text", "label"])
    raw = df["label"].astype(str).str.strip()
    df["label"] = raw.map(lambda x: label_map.get(x, numeric_map.get(x))).astype(int)
    return df["text"].tolist(), df["label"].tolist()

train_texts, train_labels = load_smsa("train")
val_texts, val_labels = load_smsa("valid")
print(f"Train: {len(train_texts)} | Val: {len(val_texts)}")

# --- 2. Tokenizer word-level sederhana + vocab dari train (self-contained) ---
def simple_tokenize(text):
    return text.lower().split()

counter = Counter(tok for t in train_texts for tok in simple_tokenize(t))
PAD, UNK = 0, 1
vocab = {"<pad>": PAD, "<unk>": UNK}
for w, _ in counter.most_common(20000):
    vocab[w] = len(vocab)
print(f"Vocab size: {len(vocab)}")

MAX_LEN = 64
def encode_text(text):
    ids = [vocab.get(t, UNK) for t in simple_tokenize(text)][:MAX_LEN]
    ids += [PAD] * (MAX_LEN - len(ids))
    return ids

def make_dataset(texts, labs):
    X = torch.tensor([encode_text(t) for t in texts], dtype=torch.long)
    y = torch.tensor(labs, dtype=torch.long)
    return TensorDataset(X, y)

train_ds = make_dataset(train_texts, train_labels)
val_ds = make_dataset(val_texts, val_labels)

# --- 3. Classifier teks ringan: embedding -> mean-pool -> head ---
# encode() menghasilkan satu vektor KONTINU per kalimat; di sinilah mixup bisa diterapkan.
class TextMeanClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=100, hidden=128, num_classes=3, pad_idx=PAD):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.head = nn.Sequential(
            nn.Linear(embed_dim, hidden),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden, num_classes),
        )
        self.pad_idx = pad_idx

    def encode(self, input_ids):
        mask = (input_ids != self.pad_idx).float().unsqueeze(-1)   # (B, T, 1)
        emb = self.embed(input_ids)                                # (B, T, E)
        pooled = (emb * mask).sum(1) / mask.sum(1).clamp(min=1)     # (B, E) - kontinu
        return pooled

    def classify(self, pooled):
        return self.head(pooled)

    def forward(self, input_ids):
        return self.classify(self.encode(input_ids))

def set_seed(seed):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        pred = model(x).argmax(1)
        correct += (pred == y).sum().item()
        total += y.size(0)
    return correct / max(total, 1)

print("Setup selesai. device:", device)

## 1. Pseudocode manual SEBELUM minta ke LLM

**Ini wajib dilakukan dulu.** Tulis algoritma dalam bahasa kamu sendiri, tanpa kode. Tujuannya: kamu bisa mendeteksi jika LLM memberikan implementasi yang berbeda dari yang kamu maksud.

Referensi: Zhang et al. 2018, "mixup: Beyond Empirical Risk Minimization" (arXiv:1710.09412).

> [!IMPORTANT]
> Mixup asalnya teknik citra: ia menginterpolasi dua input secara linear, `x = lam*x1 + (1-lam)*x2`. Pada teks, input model adalah token ID diskret. Sel di bawah menunjukkan kenapa mixup tidak bisa diterapkan langsung di token, sehingga kita menerapkannya di vektor embedding (lihat W7 §1.2-§1.3).

In [ ]:
# Ilustrasi: kenapa mixup TIDAK bisa diterapkan langsung pada token ID.
ids_a = torch.tensor([5, 12, 99, 0, 0])   # kalimat A sebagai token ID diskret
ids_b = torch.tensor([7, 3, 41, 8, 0])    # kalimat B
lam = 0.6
mixed_ids = lam * ids_a + (1 - lam) * ids_b
print("Hasil 'mixup' token ID:", mixed_ids.tolist())
print("-> ID pecahan/tak bermakna; angka ini tidak menunjuk token mana pun di vocab.")
print("Kesimpulan: mixup butuh input kontinu, jadi terapkan di vektor embedding (model.encode), bukan token.")

### Pseudocode Mixup (isi sendiri)

```
Fungsi mixup_batch(x, y, alpha):   # x = vektor embedding kontinu (B, E), bukan token ID
  1. [tulis langkah 1]
  2. [tulis langkah 2]
  3. [dst.]
  Output: [apa yang dikembalikan]
```

## 2. Prompt yang dipakai ke LLM

Tempel prompt *persis* yang kamu kirimkan. Bukan rekonstruksi. Bukan rangkuman. Copy-paste.
Ini penting untuk traceability - jika implementasi ada bug, kita bisa melihat apakah bug berasal dari prompt yang kurang jelas. Sertakan di prompt bahwa mixup akan diterapkan pada vektor embedding (B, E), bukan token.

```
[TEMPEL PROMPT DI SINI]
```

## 3. Output LLM (verbatim, sebelum modifikasi)

Tempel output asli LLM sebagai komentar kode atau markdown. Ini adalah *baseline* untuk membandingkan perubahan yang kamu buat. Perhatikan: LLM sering memberi mixup gaya citra (mengoperasikan `x` 4-dimensi). Catat apakah kamu perlu mengubahnya agar bekerja pada embedding (B, E).

```python
# [OUTPUT LLM VERBATIM DI SINI]
# (salin seluruh fungsi yang diberikan LLM, tanpa mengubah satu pun karakter)
```

## 4. Implementasi yang sudah diverifikasi dan dimodifikasi

`mixup_batch` di sini bersifat generik: ia mencampur tensor fitur kontinu apa pun. Pada lab ini kita memberinya vektor embedding hasil `model.encode`, bukan token ID.

In [ ]:
import numpy as np
import torch

def mixup_batch(
    x: torch.Tensor,
    y: torch.Tensor,
    alpha: float = 0.2
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, float]:
    """Mixup augmentation (Zhang et al. 2018) pada fitur kontinu.

    Args:
        x: tensor fitur kontinu, mis. embedding kalimat (B, E). BUKAN token ID.
        y: label (B,)
        alpha: parameter distribusi Beta.

    Returns:
        mixed_x: fitur yang sudah dicampur
        y_a: label asli
        y_b: label hasil permutasi
        lam: koefisien pencampuran (1 = semua y_a, 0 = semua y_b)
    """
    if alpha <= 0.0:
        return x, y, y, 1.0

    lam = float(np.random.beta(alpha, alpha))
    index = torch.randperm(x.size(0), device=x.device)

    mixed_x = lam * x + (1 - lam) * x[index]
    y_a = y
    y_b = y[index]
    return mixed_x, y_a, y_b, lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    """Loss mixup: lam * L(pred, y_a) + (1-lam) * L(pred, y_b)."""
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


print('Fungsi mixup_batch dan mixup_criterion didefinisikan.')

## 5. Sanity tests (wajib sebelum eksperimen)

Karena mixup bekerja di level embedding, tensor uji berbentuk (B, E) yang meniru keluaran `model.encode`, bukan tensor citra.

In [ ]:
torch.manual_seed(0)

EMB = 100
x = torch.randn(8, EMB)   # 8 vektor embedding kalimat (meniru model.encode)
y = torch.tensor([0, 1, 2, 0, 1, 2, 0, 1])

# Test 1: alpha=0 harus pass-through
mx, ya, yb, lam = mixup_batch(x, y, alpha=0.0)
assert torch.allclose(mx, x), 'alpha=0 harus mengembalikan x tanpa perubahan'
assert lam == 1.0, 'alpha=0 harus lam=1.0'
assert (ya == yb).all(), 'alpha=0: y_a harus sama dengan y_b'
print('Test 1 lulus: alpha=0 -> pass-through')

# Test 2: alpha>0, lam harus dalam [0, 1]
lams = [mixup_batch(x, y, alpha=1.0)[3] for _ in range(100)]
assert all(0 <= l <= 1 for l in lams), 'lam harus selalu dalam [0, 1]'
print(f'Test 2 lulus: lam selalu dalam [0,1]. Mean lam: {np.mean(lams):.3f} (harapan ~0.5 untuk alpha=1)')

# Test 3: shape embedding harus terjaga
mx, _, _, lam = mixup_batch(x, y, alpha=0.5)
assert mx.shape == x.shape, f'Shape harus sama. Got: {mx.shape}'
print(f'Test 3 lulus: output shape benar {tuple(mx.shape)}. lam={lam:.3f}')

# Test 4: loss dengan mixup criterion (logits 3 kelas)
model_logits = torch.randn(8, 3)
criterion = torch.nn.CrossEntropyLoss()
loss_normal = criterion(model_logits, y).item()
mx, ya, yb, lam = mixup_batch(x, y, alpha=0.5)
loss_mixup = mixup_criterion(criterion, model_logits, ya, yb, lam).item()
print(f'Test 4 lulus: loss_normal={loss_normal:.4f}, loss_mixup={loss_mixup:.4f}')

print('\nSemua sanity test lulus. Lanjutkan ke eksperimen.')

## 6. Eksperimen baseline vs mixup (level embedding) pada SmSA

Mixup disisipkan tepat di antara `model.encode` (menghasilkan vektor kalimat kontinu) dan `model.classify` (head). Inilah penerapan yang benar untuk teks: token tidak dicampur, vektor embedding-nya yang dicampur.

In [ ]:
def train_with_mixup(seed, use_mixup=False, alpha=0.2, epochs=15, lr=1e-3):
    set_seed(seed)
    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=128)

    model = TextMeanClassifier(len(vocab)).to(device)
    crit = nn.CrossEntropyLoss()
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    best_val = 0.0
    for epoch in range(epochs):
        model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            pooled = model.encode(x)                    # (B, E) - vektor kontinu
            if use_mixup:
                pooled, y_a, y_b, lam = mixup_batch(pooled, y, alpha=alpha)
                logits = model.classify(pooled)
                loss = mixup_criterion(crit, logits, y_a, y_b, lam)
            else:
                logits = model.classify(pooled)
                loss = crit(logits, y)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
        best_val = max(best_val, evaluate(model, val_loader))
    return best_val


RUN_EXPERIMENT = True   # model ringan; aman dijalankan langsung setelah sanity test lulus
seeds = [42, 123]
results = {"baseline": [], "mixup": []}

if RUN_EXPERIMENT:
    for seed in seeds:
        t0 = time.time()
        b = train_with_mixup(seed, use_mixup=False)
        results["baseline"].append(b)
        print(f"seed {seed} - baseline val_acc={b:.4f} ({time.time() - t0:.0f}s)")

        t0 = time.time()
        m = train_with_mixup(seed, use_mixup=True, alpha=0.2)
        results["mixup"].append(m)
        print(f"seed {seed} - mixup    val_acc={m:.4f} ({time.time() - t0:.0f}s)")
else:
    print("RUN_EXPERIMENT=False. Aktifkan setelah sanity tests lulus.")

## 7. Ringkas hasil baseline vs mixup

In [ ]:
if results["baseline"] and results["mixup"]:
    baseline_arr = np.array(results["baseline"])
    mixup_arr = np.array(results["mixup"])

    print("=== Hasil Baseline vs Mixup embedding (15 epoch, 2 seed, SmSA) ===")
    print(f"Baseline: {baseline_arr * 100} -> mean={baseline_arr.mean() * 100:.2f}%, std={baseline_arr.std() * 100:.2f}%")
    print(f"Mixup:    {mixup_arr * 100} -> mean={mixup_arr.mean() * 100:.2f}%, std={mixup_arr.std() * 100:.2f}%")
    delta = (mixup_arr.mean() - baseline_arr.mean()) * 100
    print(f"Delta mixup - baseline: {delta:+.2f}%")

    fig, ax = plt.subplots(figsize=(6, 4))
    bar_labels = ["Baseline", "Mixup embedding alpha=0.2"]
    means = [baseline_arr.mean() * 100, mixup_arr.mean() * 100]
    stds = [baseline_arr.std() * 100, mixup_arr.std() * 100]
    bars = ax.bar(bar_labels, means, yerr=stds, capsize=8, color=["steelblue", "tomato"], alpha=0.85, edgecolor="white")
    for bar, mean, std in zip(bars, means, stds):
        ax.text(bar.get_x() + bar.get_width() / 2, mean + std + 0.3, f"{mean:.1f}+/-{std:.1f}", ha="center", va="bottom", fontsize=10)

    ax.set_ylabel("Best Val Accuracy (%) - mean +/- std, 2 seed")
    ax.set_title("Lab W7: Baseline vs Mixup embedding (SmSA)")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()

    plot_dir = ROOT / "experiments" / "plots"
    plot_dir.mkdir(parents=True, exist_ok=True)
    plt.savefig(plot_dir / "lab_w7_mixup_text.png", dpi=120, bbox_inches="tight")
    plt.show()
else:
    print("Belum ada hasil. Jalankan eksperimen setelah RUN_EXPERIMENT=True.")

## 8. Update `docs/llm_log.md`

In [ ]:
log_path = ROOT / "docs" / "llm_log.md"
log_path.parent.mkdir(parents=True, exist_ok=True)

entry_template = f"""
---

## {date.today()} - Lab W7: Mixup Augmentation (level embedding, SmSA)

**Tool:** [Claude / ChatGPT / Copilot - pilih yang dipakai]
**Goal:** Implementasi `mixup_batch` untuk classifier sentimen teks SmSA, diterapkan di vektor embedding (bukan token ID).

**Prompt:**
```
[tempel di sini]
```

**Apa yang diubah dari output LLM:**
- [mis. LLM memberi mixup gaya citra; saya ubah agar bekerja pada embedding (B, E)]
- [perubahan 2]

**Sanity tests yang dijalankan:**
- alpha=0 -> pass-through: LULUS/GAGAL
- lam dalam [0,1]: LULUS/GAGAL
- shape embedding terjaga: LULUS/GAGAL
- mixup_criterion benar: LULUS/GAGAL

**Catatan:**
[apa yang berguna dari LLM dan apa yang harus diperbaiki sendiri]
"""

if not log_path.exists():
    log_path.write_text("# LLM Interaction Log\n" + entry_template, encoding="utf-8")
    print(f"Dibuat: {log_path.relative_to(ROOT)}")
else:
    with log_path.open("a", encoding="utf-8") as f:
        f.write(entry_template)
    print(f"Diupdate: {log_path.relative_to(ROOT)}")

## 9. Refleksi

1. Apa bagian output LLM yang paling berbeda dari pseudocode-mu? Apakah perbedaannya adalah *bug*, *perbedaan desain*, atau *hal yang tidak kamu pikirkan sebelumnya*?

2. Jika sanity test `alpha=0` gagal, apa yang itu beritahu tentang implementasi? Apa kemungkinan penyebabnya?

3. Kenapa mixup diterapkan pada vektor embedding, bukan pada token ID? Apa yang terjadi pada makna input jika kamu memaksakan `lam*ids_a + (1-lam)*ids_b` (lihat ilustrasi §1)?

4. Mixup biasanya membantu regularisasi. Jika hasilmu menunjukkan mixup *tidak* membantu atau bahkan menurunkan akurasi pada 15 epoch, apa hipotesismu? Eksperimen apa yang akan menjawabnya?

### Jawaban Refleksi

**1. Perbedaan terbesar dari pseudocode:**
> *[tulis di sini]*

**2. Jika alpha=0 gagal:**
> *[tulis di sini]*

**3. Kenapa di embedding, bukan token:**
> *[tulis di sini]*

**4. Jika mixup tidak membantu:**
> *[tulis di sini]*

## Self-Check Quick

- [ ] Pseudocode `mixup_batch` ditulis sendiri SEBELUM minta ke LLM (bukan setelahnya).
- [ ] Prompt LLM verbatim disalin ke notebook (bukan rekonstruksi dari memori).
- [ ] Output LLM verbatim disalin sebagai baseline modifikasi (sebelum diubah).
- [ ] 4 sanity test lulus: alpha=0 pass-through, lam dalam [0,1], shape embedding terjaga, mixup_criterion benar.
- [ ] Mixup diterapkan di level embedding (`model.encode` -> mixup -> `model.classify`), bukan di token ID.
- [ ] Eksperimen baseline vs mixup dijalankan dengan 2 seed x 15 epoch pada SmSA.
- [ ] `docs/llm_log.md` diupdate dengan entri lab ini.